In [1]:
# 1. INSTALL DEPENDENCIES
!pip install av torchvision

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
from torchvision.io import read_video
from google.colab import drive

# --- 2. CONFIGURATION ---
drive.mount('/content/drive', force_remount=True)

DATA_PATH = "/content/drive/MyDrive/video_data/train"
TEST_VIDEO = "/content/drive/MyDrive/video_data/test/unknown.avi"
CLASSES = ['walking', 'running']
BATCH_SIZE = 2
NUM_FRAMES = 16
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. DATASET DEFINITION ---
class VideoDataset(Dataset):
    def __init__(self, root, classes, transform=None, num_frames=16):
        self.samples = []
        self.transform = transform
        self.num_frames = num_frames
        for i, cls_name in enumerate(classes):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path): continue
            for f in os.listdir(cls_path):
                if f.lower().endswith(('.mp4', '.avi', '.mov')):
                    self.samples.append((os.path.join(cls_path, f), i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            # vframes shape: (T, C, H, W)
            vframes, _, _ = read_video(path, pts_unit='sec', output_format='TCHW')
            t = vframes.shape[0]
            if t == 0: raise ValueError("Empty Video")

            # Uniform sampling of frames
            indices = torch.linspace(0, t - 1, self.num_frames).long()
            vframes = vframes[indices]

            # Swin3D expects (C, T, H, W)
            vframes = vframes.permute(1, 0, 2, 3)
            if self.transform:
                vframes = self.transform(vframes)
            return vframes, label
        except:
            # Fallback for corrupted files
            return torch.zeros((3, self.num_frames, 224, 224)), label

# --- 4. MODEL INITIALIZATION ---
weights = Swin3D_T_Weights.KINETICS400_V1
model = swin3d_t(weights=weights)
model.head = nn.Linear(model.head.in_features, len(CLASSES))
model = model.to(DEVICE)

train_transforms = weights.transforms()
train_dataset = VideoDataset(DATA_PATH, CLASSES, transform=train_transforms, num_frames=NUM_FRAMES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- 5. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print(f"\n--- Starting Training ({len(train_dataset)} videos) ---")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for i, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(DEVICE), labels.to(DEVICE)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "video_swin_model.pth")
print("Training Complete. Model Saved.")

# --- 6. PREDICTION ON UNKNOWN.AVI ---
print(f"\n--- Predicting for: {TEST_VIDEO} ---")
model.eval()

def predict_single_video(path):
    try:
        vframes, _, _ = read_video(path, pts_unit='sec', output_format='TCHW')
        indices = torch.linspace(0, vframes.shape[0] - 1, NUM_FRAMES).long()
        vframes = vframes[indices].permute(1, 0, 2, 3)

        input_tensor = train_transforms(vframes).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        return CLASSES[pred_idx.item()], conf.item()
    except Exception as e:
        return f"Error: {str(e)}", 0.0

label, confidence = predict_single_video(TEST_VIDEO)
print(f"RESULT: {label.upper()} ({confidence*100:.2f}% confidence)")

Mounted at /content/drive

--- Starting Training (10 videos) ---


/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Epoch [1/3] Average Loss: 0.7002
Epoch [2/3] Average Loss: 0.5517
Epoch [3/3] Average Loss: 1.2850
Training Complete. Model Saved.

--- Predicting for: /content/drive/MyDrive/video_data/test/unknown.avi ---
RESULT: ERROR: INDEX IS OUT OF BOUNDS FOR DIMENSION WITH SIZE 0 (0.00% confidence)


In [11]:
# --- TARGET FILE ---
NEW_VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"

import torch
import torch.nn.functional as F
from torchvision.io import read_video
import torchvision.transforms as T
import os

def run_guaranteed_inference(video_path):
    model.eval()

    try:
        # 1. Load video. 'TCHW' returns (Time, Channels, Height, Width)
        vframes, _, _ = read_video(video_path, pts_unit='sec', output_format='TCHW')

        # 2. Sample 16 frames
        t = vframes.shape[0]
        indices = torch.linspace(0, t - 1, 16).long()
        vframes = vframes[indices] # Shape: (16, 3, H, W)

        # 3. Resize and Normalize manually to avoid the "16 vs 3" error
        # We process each frame individually to keep the math straight
        preprocess = T.Compose([
            T.Resize((224, 224)),
            T.ConvertImageDtype(torch.float32),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Apply preprocess to the (Time, Channels, H, W) tensor
        # This treats the first dim (16) as the batch for the transform
        vframes = preprocess(vframes)

        # 4. NOW permute for the Swin Model: (T, C, H, W) -> (C, T, H, W)
        # Required shape: (3, 16, 224, 224)
        vframes = vframes.permute(1, 0, 2, 3)

        # 5. Add Batch Dimension and move to GPU
        input_tensor = vframes.unsqueeze(0).to(DEVICE)

        # 6. Inference
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")
        print(f"PREDICTED: {CLASSES[pred_idx.item()].upper()}")
        print(f"CONFIDENCE: {conf.item()*100:.2f}%")
        print("-" * 30)

    except Exception as e:
        print(f"Inference failed: {e}")

# Run it
run_guaranteed_inference(NEW_VIDEO_PATH)

------------------------------
FILE: v_PlayingCello_g04_c02.avi
PREDICTED: WALKING
CONFIDENCE: 55.84%
------------------------------
